# Training Dual-Branch Model with Kaggle Data

This notebook trains the dual-branch malware detection model using the preprocessed Kaggle datasets.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
from tqdm import tqdm

from models import DualBranchModel
from utils import (
    create_data_loaders,
    calculate_metrics, 
    plot_confusion_matrix, 
    plot_roc_curve,
    plot_training_history,
    print_classification_report
)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Load Preprocessed Data

In [ ]:
def load_processed_data(data_path='../data/processed'):
    """Load preprocessed data from disk"""
    data_path = Path(data_path)
    
    data = {
        'train_cnn': np.load(data_path / 'train_cnn.npy'),
        'train_lstm': np.load(data_path / 'train_lstm.npy'),
        'train_labels': np.load(data_path / 'train_labels.npy'),
        
        'val_cnn': np.load(data_path / 'val_cnn.npy'),
        'val_lstm': np.load(data_path / 'val_lstm.npy'),
        'val_labels': np.load(data_path / 'val_labels.npy'),
        
        'test_cnn': np.load(data_path / 'test_cnn.npy'),
        'test_lstm': np.load(data_path / 'test_lstm.npy'),
        'test_labels': np.load(data_path / 'test_labels.npy'),
    }
    
    with open(data_path / 'vocab.pkl', 'rb') as f:
        data['vocab'] = pickle.load(f)
    
    data['vocab_size'] = len(data['vocab'])
    
    return data

# Load data
data = load_processed_data()

print("Data loaded successfully!")
print(f"Training samples: {len(data['train_labels'])}")
print(f"Validation samples: {len(data['val_labels'])}")
print(f"Test samples: {len(data['test_labels'])}")
print(f"Vocabulary size: {data['vocab_size']}")

## 3. Create Data Loaders

In [ ]:
batch_size = 32

train_loader, val_loader = create_data_loaders(
    data['train_cnn'], data['train_lstm'], data['train_labels'],
    data['val_cnn'], data['val_lstm'], data['val_labels'],
    batch_size=batch_size,
    num_workers=0,  # Set to 0 for Windows
    shuffle=True
)

print(f'Training batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')

## 4. Initialize Model

Configure the model based on your data dimensions:

In [ ]:
# Get data dimensions
cnn_channels = data['train_cnn'].shape[1]  # Should be 1 for grayscale
lstm_input_size = data['vocab_size']  # Use vocab size as input dimension
seq_length = data['train_lstm'].shape[1]

print(f"CNN input channels: {cnn_channels}")
print(f"LSTM input size: {lstm_input_size}")
print(f"Sequence length: {seq_length}")

# Initialize model
model = DualBranchModel(
    cnn_config={
        'input_channels': cnn_channels,
        'num_classes': 2,
        'feature_dim': 256
    },
    lstm_config={
        'input_size': lstm_input_size,  # Embedding dimension
        'hidden_size': 128,
        'num_layers': 2,
        'feature_dim': 256,
        'bidirectional': True
    },
    num_classes=2,
    fusion_method='concat'  # Try 'concat', 'add', or 'attention'
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Training Configuration

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

# Training parameters
num_epochs = 20
best_val_loss = float('inf')
best_val_acc = 0.0
patience = 7
patience_counter = 0

## 6. Training Loop with Early Stopping

In [ ]:
history = {
    'train_loss': [],
    'train_accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

for epoch in range(num_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*60}")
    
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    train_pbar = tqdm(train_loader, desc='Training')
    for batch in train_pbar:
        cnn_input = batch['cnn_input'].to(device)
        lstm_input = batch['lstm_input'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(cnn_input, lstm_input)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        # Update progress bar
        train_pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * train_correct / train_total:.2f}%'
        })
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc='Validation')
        for batch in val_pbar:
            cnn_input = batch['cnn_input'].to(device)
            lstm_input = batch['lstm_input'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(cnn_input, lstm_input)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
            val_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * val_correct / val_total:.2f}%'
            })
    
    # Calculate epoch metrics
    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total
    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    # Update history
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    
    # Print epoch summary
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, '../checkpoints/best_model.pth')
        print(f"  ✓ Best model saved! (Val Acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\n⚠️ Early stopping triggered after {epoch+1} epochs")
        break

print(f"\n{'='*60}")
print("Training completed!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"{'='*60}")

## 7. Plot Training History

In [ ]:
plot_training_history(history, metrics=['loss', 'accuracy'])

## 8. Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load('../checkpoints/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Create test loader
from utils.data_loader import MalwareDataset
from torch.utils.data import DataLoader

test_dataset = MalwareDataset(data['test_cnn'], data['test_lstm'], data['test_labels'])
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Get predictions
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        cnn_input = batch['cnn_input'].to(device)
        lstm_input = batch['lstm_input'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(cnn_input, lstm_input)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

## 9. Calculate and Display Metrics

In [ ]:
metrics = calculate_metrics(all_labels, all_preds, all_probs)

print('\n' + '='*60)
print('TEST SET EVALUATION METRICS')
print('='*60)
print(f"Accuracy:    {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
print(f"Precision:   {metrics['precision']:.4f}")
print(f"Recall:      {metrics['recall']:.4f}")
print(f"F1-Score:    {metrics['f1_score']:.4f}")
print(f"AUC:         {metrics['auc']:.4f}")
print(f"Specificity: {metrics['specificity']:.4f}")
print(f"Sensitivity: {metrics['sensitivity']:.4f}")
print('='*60)

print_classification_report(all_labels, all_preds)

## 10. Visualize Results

In [ ]:
# Confusion Matrix
plot_confusion_matrix(all_labels, all_preds, class_names=['Benign', 'Malware'])

In [ ]:
# ROC Curve
plot_roc_curve(all_labels, all_probs)

## 11. Save Final Results

In [ ]:
# Save metrics
import json

results = {
    'test_metrics': {
        'accuracy': float(metrics['accuracy']),
        'precision': float(metrics['precision']),
        'recall': float(metrics['recall']),
        'f1_score': float(metrics['f1_score']),
        'auc': float(metrics['auc']),
        'specificity': float(metrics['specificity']),
        'sensitivity': float(metrics['sensitivity'])
    },
    'training_history': history,
    'model_config': {
        'cnn_channels': cnn_channels,
        'lstm_input_size': lstm_input_size,
        'fusion_method': 'concat',
        'num_epochs_trained': len(history['train_loss']),
        'best_val_acc': float(best_val_acc),
        'best_val_loss': float(best_val_loss)
    }
}

with open('../checkpoints/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Results saved to checkpoints/training_results.json")

## Summary

🎉 **Training Complete!**

Your dual-branch malware detection model has been successfully trained on the Kaggle datasets.

**Next Steps:**
- Experiment with different fusion methods ('add', 'attention')
- Try different hyperparameters (learning rate, batch size, etc.)
- Implement data augmentation
- Perform cross-validation
- Deploy the model for inference